In [110]:
import numpy as np 
import k3d

### Алгоритм размещения фотоумножителей

- Средняя площадь поверхности на один ФЭУ:
  $$
  S_0=\frac{4\pi R^2}{N}.
  $$

- Характерный линейный размер ячейки:
  $$
  l\approx\sqrt{S_0}.
  $$

- Сфера разбивается на широтные слои с угловым шагом
  $$
  \Delta\theta\approx \frac{l}{R}.
  $$

- Число ФЭУ в слое выбирается пропорционально длине соответствующей параллели:
  $$
  n_\varphi(\theta)\propto \sin\theta.
  $$

- После оценки непрерывного распределения $n_\varphi\propto\sin\theta$ было выбрано симметричное целочисленное распределение $(2,7,11,11,7,2)$, обеспечивающее меньший разброс расстояний до ближайших соседей.

- Внутри каждого слоя ФЭУ размещаются равномерно по углу $\varphi$.

- Соседние слои сдвигаются относительно друг друга на:
  $$
  \Delta\varphi_{phase}
  =
  \frac{\pi}{\operatorname{lcm}(n_i,n_{i+1})},
  $$
  чтобы увеличить минимальное расстояние между ФЭУ соседних слоёв.

- Для контроля равномерности вычисляется расстояние от каждого ФЭУ до ближайшего соседа.

In [111]:
class Detector():
    def __init__(self, R: float, N: int, r: float, a: float):
        """
        Parameters
        ----------
        R : float
            Радиус детектора в метрах.
        N : int
            Количество фотоумножителей.
        r : float
            Радиус фотокатода в метрах.
        a : float
            Эффективность фотокатода.
        """
        self.R = R
        self.N = N
        self.r = r
        self.a = a

    def coord_cart(self, theta: float, phi: float) -> np.array:
        return np.array([self.R * np.sin(theta)*np.cos(phi), self.R * np.sin(theta)*np.sin(phi), self.R*np.cos(theta)])

    def fill_detector(self):
        self.photo_multipliers = np.zeros((self.N, 3))
        S = 4 * self.R**2 * np.pi 
        delta_S = S / self.N 
        lin_size = delta_S**0.5 
        N_theta = np.pi * R / lin_size 
        delta_theta = np.pi / N_theta

        theta_floor = [int(np.floor(N_theta)), np.pi / np.floor(N_theta), delta_theta - np.pi / np.floor(N_theta)]
        theta_ceil = [int(np.ceil(N_theta)), np.pi / np.ceil(N_theta), delta_theta - np.pi / np.ceil(N_theta)]

        best_theta = min(theta_floor, theta_ceil, key=lambda x: abs(x[2]))
        N_theta, delta_theta = best_theta[0], best_theta[1]

        mid_thetas = np.array([delta_theta / 2 + delta_theta*i for i in range(0, N_theta)])
        n_phi = np.array([(np.pi * self.N)**0.5*np.sin(theta) for theta in mid_thetas])

        A = self.N / sum([np.sin(theta) for theta in mid_thetas])
        #n_phi = np.rint(np.array([A * np.sin(theta) for theta in mid_thetas])) #-> (3, 7, 10, 10, 7, 3)
        n_phi = np.array([2, 7, 11, 11, 7, 2])     

        phi_phases = np.zeros(N_theta)
        for i in range(N_theta):
            if i == 0:
                continue
            phi_phases[i] = np.pi / np.lcm(int(n_phi[i-1]), int(n_phi[i]))

        n = 0
        phase = 0
        for i in range(N_theta):
            if i < 3:
                phase += phi_phases[i]
            else:
                phase -= phi_phases[i]
            for j in range(int(n_phi[i])):
                self.photo_multipliers[n] = self.coord_cart(mid_thetas[i], 2*np.pi/n_phi[i] * j + phase)
                n += 1

    def in_chord(self, coord: np.array) -> int:
        for k, photo_multiplier in enumerate(self.photo_multipliers):
            distance = np.linalg.norm(photo_multiplier-coord)
            if distance <= self.r:
                return k
        return -1

In [112]:
R = 2
N = 40
r = 0.1
a = 0.5

detector = Detector(R, N, r, a)
detector.fill_detector()

test_coordinates = np.array([[-1.37, 0.31, 1.41],[2, 0, 0]])
print(f'point {test_coordinates[0]} in multiplier number {detector.in_chord(test_coordinates[0])}')
print(f'point {test_coordinates[1]} in multiplier number {detector.in_chord(test_coordinates[1])}')

min_distances = np.zeros(detector.N)
for k, multiplier_1 in enumerate(detector.photo_multipliers):
    current_min = 1000
    for l, multiplier_2 in enumerate(detector.photo_multipliers):
        if k == l:
            continue
        distance = np.linalg.norm(multiplier_1-multiplier_2)
        if distance < current_min:
            current_min = distance
    min_distances[k] = current_min

print(max(min_distances), min(min_distances), sum(min_distances)/detector.N, np.linalg.norm(min_distances)**2/detector.N - (sum(min_distances)/detector.N)**2)


point [-1.37  0.31  1.41] in multiplier number 5
point [2. 0. 0.] in multiplier number -1
1.1371593019546309 1.035276180410083 1.0669063111709831 0.0007463853248603769


In [113]:
plot = k3d.plot()
points = k3d.points(detector.photo_multipliers,point_size=0.2)

plot += points
plot.display()

Output()